## **What exactly is WBGAPI**

`wbgapi` is a Python package for accessing data from the **World Bank API**.That makes retrieving economic/development data much easier than manually constructing API URLs.

> **World Bank database -> World Bank API -> WBGAPI -> Python -> pandas -> Analysis/Visualization**

Install:
```bash
# Bash
pip install wbgapi
```
Import:
```python
import wbgapi as wb # The documentation commonly uses `wb` as the namespace.
```

### **The most important mental model**

Before learning commands, understand the **four things we are normally asking the World Bank for:**

```text
WORLD BANK DATA
|
Indicators "What" -> (GDP, Population, Inflation)
|
Economy "Where" -> (China, USA, India)
|
Time "When" -> (2010, 2011, 2012)
```
And there is one more important dimension:
```text
Source / Database
"Which WORLD BANK DATABASE?"
```
For example, WDI = World Development Indicators.

WBGAPI organizes these concepts into objects such as `series`, `economy`, `time`, `source`, `region`, `income`, `lending`, and `topic`.

This is the foundation of the entire package.

### **The four most important WBGAPI objects**

#### **A. `wb.series`**
This represents **indicators**.

Think:
> "What economic variable do I want?"

Example:
```text
GDP
Population
Inflation
Unemployment
CO2 emissions
Life expectancy
```
Each indicator has a code.

For example:
```python
NY.GDP.MKTP.CD
```
means GDP in current US dollars.

we can search:
```python
wb.series.info(q='GDP')
```
or:
```python
wb.series.info(q='population')
```
The documentation describes the `info()` functions as one of the primary ways to search the API.

#### **B. `wb.economy`**
The represents **countries/economies**.

For example:
```text
IND -> India
USA -> United States
CHN -> China
GBR -> United Kingdom
```
we can inspect India:
```python 
wb.economy.info('IND')
```
Or search:
```python
wb.economy.info(q='India')
```
We can also work with multiple economies:
```python
wb.economy.info(['IND', 'USA', 'CHN'])
```
The cookbook demonstrates this pattern directly.

**_Important distinction_**

WBGAPI uses the word _economy_, not simply "country".

Why?

Because World Bank databases can contain aggregates and, depending on the database, subnational entities as well.

So mentally:
```text
Economy
    - Countries
    - Regions
    - Income groups / aggregates
    - Other entities supported by a database
```

### **C. `wb.time`**

This handles **time periods**.

For example:
```python
range(2010, 2021)
```
means:
```text
2010 - 2011 - 2012 - ..... 2020
```
We can therefore tell WBGAPI:
> "Give me India's GDP from 2010 - 2020.

This "indicator + economy + time" structure is probably the single most important thing to understand.

### **D. `wb.source`**

This represents the **World Bank databases/sources**.

For example:
```python
wb.source.info()
```
can be used to see available databases.

WBGAPI normally uses the **World Development Indicators (WDI)** databae, whose database ID is `2`. We can change the database globally or for an individual request.

For example:
```python
wb.db = 2
```
This becomes important when we start working with datasets  beyond the standard WDI.

### **The most important function: `wb.data`**

Now we get to the part we'll use most as a data analyst.

Suppose we want:

> India's GDP from 2010 to 2023. 

We can use:
```python
df = wb.data.DataFrame(
    "NY.GDP.MKTP.CD",
    "IND",
    time=range(2010, 2024)
)
```
And now:
```python
df
```
gives us a pandas DataFrame.

The cookbook and current package documentation show `DataFrame()` as the convenient pandas-oriented approach.

This is where WBGAPI becomes extremely userul for **data analysis**.

### **Multiple countries**
We don't have to request countries one at a time.
```python
df = wb.data.DataFrame(
    "NY.GDP.MKTP.CD",
    ['IND', 'USA', 'CHN'],
    time=range(2010, 2024)
)
```

WBGAPI is designed to allow multipe series, economies and time periods in a single request.

### **Multiple Indicators**
We can also request several indicators.

For example:
```python
df = wb.data.DataFrame(
    ['NY.GDP.MKTP.CD', 'SP.POP.TOTL'],
    ['IND','CHN'],
    time=range(2010, 2024)
)
```
Now our conceptual dataset becomes:
```text
Country x Year x Indicator
```
This is powerful because we can start asking analytical questions such as:
> Does GDP growth correspond with population growth?
or:
> How does GDP per capita differ between India and the USA?

### **`info` vs `list()` vs `get()` vs `Series()`**

This is an important part of the documentation.

WBGAPI's feature objects generally provide four common methods:

| **Method** | **Purpose** |
|------------|-------------|
| `list()` | Get iterable collection of elements |
| `get()` | Get one specific element |
| `info()` | Human-readable information/search |
| `Series()` | Return elements as a pandas Series |

The documentation explicitly describes this common interface.

Think of them like this:
```text
info()  → "Show me what exists."
list()  → "Give me the elements so Python can process them."
get()   → "Give me this specific element."
Series()→ "Give me the elements as pandas Series."
```

### **Searching is a major feature**

Suppose we don't know the indicator code.

We only know:
> "I need CO2 data."

Instead of Googling the indicator code, we can do:
```python
wb.series.info(q='co2')
```
WBGAPI searches the World Bank's indicator metadata and returns matching indicators.

for example, we might discover indicators such as:
```text
EN.ATM.CO2E.KT
EN.ATM.CO2E.PC
...
```

### `wb.search()`
There is also a broader search function:

```python
wb.search('pound')
```
The cookbook notes that this searches across available concepts and returns a print friendly object. Its results can also be accessed programtically through. `.metadata`.

So we can think:
```text 
series.info() -> Search specifically for indicators

wb.search() -> Broader World Bank metadata search
```

### **Metadata -- Very important**

Suppose we find:
```python
NY.GDP.MKTP.Cd
```
we shouldn't immediately assume we understand everything about it.

We should investigate its metadata.

For example:
```python
wb.series.metadata.get('NY.GDP.MKTP.CD')
```
WBGAPI supports metadata for series, economies and combinations.

Why does this matter?

Imagine our analysis says:
> "India's GDP increased."

we need to know:
- what exactly is GDP here?
- What unit?
- Current or constant prices?
- Which currency?
- What is the source?
- What methodology?
- Are there notes/footnotes?
- Are there missing values?

**Data retrieval without metadata can produce technically correct but conceptually wrong analysis.**

### **`wb.data.fetch()` vs `DataFrame()`**

`DataFrame()`

Use when we want data convniently inside pandas.
```python
df = wb.data.DataFrame(
    'NY.GDP.MKTP.CD',
    ['IND', 'USA'],
    time=range(2010, 2024)
)
```
Great for:
```text
EDA
pandas
statistics
Matplotlib
Seaborn
Excel Export
regression
correlation
```

`fetch()`

Retrieves rows as a **Python generator**.

For example:
```python
for row in wb.data.fetch("SP.POP.TOTL"):
    print(row)
```
The underlying documentation describes parameters such as `series`, `economy`, `time`, `mrv`, `mrnev`, `skipBlanks`, `labels`, `skipAggs`, and `numericTimeKeys`.

So:
```text
DataFrame()
    ↓
"I want analysis-ready pandas data."

fetch()
    ↓
"I want to iterate through API results myself."
```

### **Most Recent Values -- `mrv`**

This is another extremely useful feature.

for example:
```python
wb.data.DataFrame(
    'SP.POP.TOTL',
    mrv=1
)
```
meanse essentially:
> Give me the most recent available population value.

The cookbook demonstrates this approach for finding the most reent value across economies.

There is also:
```text
mrv
```
versus
```text
mrnev
```
Conceptually:
```text
mrv -> most recent observation according to the same time period

mrnev -> most recent non-empty observations
```
The distinction becomes useful when datasets contain missing observation.

### **`labels=True`**

Normally we may get codes:
```text
IND
USA
CHN
```
But sometimes we want readable names:
```text
India
United States
China
```
WBGAPI supports:
```python
labels=True
```
For example, `fetch()` can include both dimension IDs and names.

this is particularly useful when preparing data for reports.

### **Aggregates**

World Bank data doesn't only contain individual countries.

we can encounter aggregates such as:
```text
World
South Asia
East Asia & Pacific
High income
Low income
```
WBGAPI provides options such as:
```python
skipAggs=True
```
to exclude aggregate economies.

This matters enormously when doing statistical analysi.

For example, if we're calcualting.
> Average GDP growth across countries

we probably don't want:
```text
India
China
USA
World
South Asia
High income
```
all mixed together.

That would distort our analysis

### **Country-name conversion**

we don't always remembere ISO3 codes.

For example:
```text
India → IND
United States → USA
China → CHN
```

WBGAPI has an economy coder utility that can resolve common country names into World Bank codes.

Conceptually:
```python
wb.economy.coder(['India', 'South Korea'])
```
helps translate human-friendly names into API-friendly identifiers.

### **Pandas integration**

This is probably the most important reason we, as someone learning data analysis, should care about WBGAPI.

We can go directly:
```python
import wbgapi as wb
import pandas as pd

df = wb.data.DataFrame(
    "NY.GDP.MKTP.CD",
    ["IND", "USA", "CHN"],
    time=range(2010, 2024)
)
```
Then:
```python
df.head()
# followed by normal pandas methods:
df.describe()
df.isna().sum()
df.mean()
# and visualization:
df.plot()
```
The WBGAPI cookbook specifically demonstrates combining WBGAPI with pandas and Matplotlib.

### **The complete workflow we should remember**

Here's the real **WBGAPI workflow** recommended for memorizing conceptually, not syntactically:

```text
                  BUSINESS / RESEARCH QUESTION
                              ↓
                    "I need GDP data."
                              ↓
                         SEARCH
                              ↓
                  wb.series.info(...)
                              ↓
                  FIND INDICATOR CODE
                              ↓
                    NY.GDP.MKTP.CD
                              ↓
                      CHECK METADATA
                              ↓
              wb.series.metadata.get(...)
                              ↓
                  SELECT ECONOMIES
                              ↓
                     IND / USA / CHN
                              ↓
                     SELECT TIME
                              ↓
                    2010 → 2024
                              ↓
                      GET THE DATA
                              ↓
                  wb.data.DataFrame(...)
                              ↓
                         PANDAS
                              ↓
                  CLEAN → EDA → ANALYZE
                              ↓
                   MATPLOTLIB / SEABORN
                              ↓
                       INSIGHT
```

### **The WBGPAI cheat sheet**

| **What we want** | **Main WBGAPI tool** |
|------------------|----------------------|
| Find World Bank databases | `wb.source` |
| Find indicators | `wb.series` |
| Find countries/economies | `wb.economy` |
| Work with time | `wb.time`|
| Search metadata | `wb.search()` |
| Get data | `wb.data` |
| get pandas DataFrame | `wb.data.DataFrame()` |
| Iterate through results | `wb.data.fetch()` |
| Get one specific object | `.get()` |
| Search/filter objects | `.info()` |
| Get iterable objects | `.list()` |
| Most recent observations | `mrv=` |
| Most recent non-empty observations | `mrnev=` |
| Include readable labels | `labels=True` |
| Remove aggregates | `skpAggs=True` |
| Get indicator metadata | `wb.series.metadata.get()` |
| Convert country names -> codes | `wb.economy.coder()` |

These functions and concepts correspond to the package's documentated design.

### The 5 things I'd focus on first

Don't try to learn the entire documentation at once.

For our data-analysis journey, master these first.

1. **Find indicators**
```python
wb.series.info(q='GDP')
```

2. **Find countries**
```python
wb.economy.info('IND')
```

3. **Retrieve data**
```python
wb.data.DataFrame(...)
```

4. **Control time**
```python
time=range(2010, 2024)
```

5. **Use pandas**
```python
df.head()
df.describe()
df.info()
df.isna().sum()
```
Once those become natural, the rest of WBGAPI becomes much easier.

One important distinction to keep in our head:
> **WEBGAPI isn't the data itself. It's the Python interface that helps us ask the World Bank for Data.**